2.1 理论计算题

已知：
输入图像尺寸：3×32×32（通道×高×宽）
卷积层：16 个卷积核，每个核大小 3×5×5
填充 padding = 2，步幅 stride = 2

1. 输出特征图尺寸

输出通道数 = 16（等于卷积核个数）

输出高度 = floor((32 + 2×2 - 5) / 2) + 1 = floor((32+4-5)/2) + 1 = floor(31/2) + 1 = 15 + 1 = 16
输出宽度 = 同样 = 16

因此输出尺寸：16×16×16

结果：16 通道 × 16 高 × 16 宽

2. 单个输出像素点的乘法次数

每个输出像素对应一个卷积核在输入上的感受野。卷积核大小 5×5，输入通道数 3，因此一个卷积核覆盖的元素个数 = 3 × 5 × 5 = 75。

每次卷积操作是卷积核与对应区域逐元素相乘并求和，乘法次数 = 75 次（每个元素乘一次，然后求和不计为乘法？通常点乘中每对元素一次乘法，75次乘法后加和）。因此单个输出通道的一个像素值需要 75 次乘法操作。

答案：75 次

In [2]:
import numpy as np

def max_pool2d(X, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    X: 输入张量，形状 (H, W) 或 (C, H, W) 或 (N, C, H, W)
    kernel_size: 池化窗口大小，整数或 (h, w)
    stride: 步幅，整数或 (h, w)
    padding: 填充，整数或 (h, w)
    """
    # 统一格式：扩展为 (N, C, H, W)
    if X.ndim == 2:
        X = X.reshape(1, 1, X.shape[0], X.shape[1])
    elif X.ndim == 3:
        X = X.reshape(1, X.shape[0], X.shape[1], X.shape[2])
    N, C, H, W = X.shape
    
    # 参数解析
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    # 填充
    X_pad = np.pad(X, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=0)
    
    # 输出尺寸
    out_h = (H + 2*ph - kh) // sh + 1
    out_w = (W + 2*pw - kw) // sw + 1
    
    # 初始化输出
    out = np.zeros((N, C, out_h, out_w))
    
    # 滑动窗口
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            window = X_pad[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2,3))
    
    # 如果输入是2D，输出也降维
    if X.ndim == 2 and out.shape[0]==1 and out.shape[1]==1:
        return out[0,0]
    elif X.ndim == 3 and out.shape[0]==1:
        return out[0]
    return out

# 测试示例
if __name__ == "__main__":
    X = np.random.randn(1, 3, 32, 32)
    out = max_pool2d(X, kernel_size=2, stride=2, padding=0)
    print("输入形状:", X.shape)
    print("池化后形状:", out.shape)

输入形状: (1, 3, 32, 32)
池化后形状: (1, 3, 16, 16)


3.1 理论计算题

已知：输入和输出通道数均为 C。

1. 单个 5×5 卷积层（无偏置）的参数量

参数量 = 卷积核大小 × 输入通道数 × 输出通道数 = 5×5 × C × C = 25 C²

结果：25 C²

2. 两个串联的 3×3 卷积层（无偏置）

第一个 3×3 卷积：参数量 = 3×3 × C × C = 9 C²  
第二个 3×3 卷积：同样 = 9 C²  
总参数量 = 9 C² + 9 C² = 18 C²

结果：18 C²

比较：两个 3×3 卷积参数量（18 C²）少于一个 5×5（25 C²），且感受野相同（5×5），同时增加非线性。

In [1]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.conv(x)

# 示例：输入通道3，输出通道16，卷积核3x3，步幅1，填充1
block = NiNBlock(3, 16, 3, stride=1, padding=1)
print(block)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print("输入形状:", x.shape)
print("输出形状:", out.shape)

NiNBlock(
  (conv): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)
输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])


4.1 理论计算题

批量归一化公式（忽略 ε=0）：
y = γ × (x - μ) / σ + β

给定 4 个样本在某一通道某一位置的值：x = [2, 4, 6, 8]
γ = 2, β = 1, ε = 0

1. 计算均值 μ
μ = (2+4+6+8)/4 = 20/4 = 5

2. 计算方差 σ²（总体方差）
σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4 = (9+1+1+9)/4 = 20/4 = 5
σ = √5 ≈ 2.236

3. 计算每个样本归一化后的值
y = 2 × (x - 5)/√5 + 1

对于 x=2: (2-5)/√5 = -3/√5 ≈ -1.3416, 乘以2得 -2.6832, 加1得 -1.6832
x=4: (4-5)/√5 = -1/√5 ≈ -0.4472, 乘2得 -0.8944, 加1得 0.1056
x=6: (6-5)/√5 = 1/√5 ≈ 0.4472, 乘2得 0.8944, 加1得 1.8944
x=8: (8-5)/√5 = 3/√5 ≈ 1.3416, 乘2得 2.6832, 加1得 3.6832

结果（保留小数）：
y1 ≈ -1.6832
y2 ≈ 0.1056
y3 ≈ 1.8944
y4 ≈ 3.6832

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.relu = nn.ReLU()
        
        self.use_1x1conv = use_1x1conv
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = None
    
    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        if self.use_1x1conv:
            shortcut = self.shortcut(x)
        else:
            shortcut = x
        out += shortcut
        out = self.relu(out)
        return out

# 示例
block = Residual(3, 16, use_1x1conv=True, stride=1)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print("输入形状:", x.shape)
print("输出形状:", out.shape)

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])


5.1 理论计算题

1. 为什么底层特征提取层用较小学习率，顶层输出层用较大学习率？

- 底层特征（如边缘、纹理）在源数据集（如ImageNet）上已经学习得很好，具有通用性，不需要大幅改动，否则可能破坏有用特征。小学习率或冻结可以保留这些特征。
- 顶层输出层需要适应新的目标类别，随机初始化或与源任务不同，因此需要较大的学习率来快速学习新类别的特征。

2. 目标数据集很小且与源数据集相似时的策略

- 冻结大部分底层卷积层，只微调最后几层全连接层（或新添加的分类层）。
- 使用非常小的学习率（如1e-5）进行微调，避免过拟合。
- 使用更强的正则化（如Dropout、权重衰减）。
- 数据增强（即使数据集小，也可通过增广扩充）。
- 使用早停法（Early Stopping）监控验证集性能。

In [4]:
import torchvision.transforms as transforms
from PIL import Image
import torch

# 定义图像增广管道
augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),  # 随机裁剪并缩放
    transforms.RandomHorizontalFlip(p=0.5),                # 50%水平翻转
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # 颜色抖动
    transforms.ToTensor()                                  # 转为张量
])

# 示例：加载一张图片并应用增广
# 假设有一个图像文件 'example.jpg'，这里用随机数组模拟
# 实际使用时：img = Image.open('example.jpg').convert('RGB')
# augmented_img = augmentation_pipeline(img)

# 为了演示，创建一个随机RGB图像（PIL格式）
from PIL import Image
import numpy as np
random_img = Image.fromarray(np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8))
augmented = augmentation_pipeline(random_img)
print("原始图像尺寸:", random_img.size)
print("增广后张量形状:", augmented.shape)  # 应为 [3, 224, 224]

原始图像尺寸: (300, 300)
增广后张量形状: torch.Size([3, 224, 224])


6.1 理论计算题

真实框 A = [10, 10, 50, 50]  (左上角x, 左上角y, 右下角x, 右下角y)
预测框 B = [30, 30, 70, 70]

1. 计算交集区域
交集左上角 x = max(10,30) = 30
交集左上角 y = max(10,30) = 30
交集右下角 x = min(50,70) = 50
交集右下角 y = min(50,70) = 50
交集宽度 = 50 - 30 = 20
交集高度 = 50 - 30 = 20
交集面积 = 20 × 20 = 400

2. 计算并集面积
A 的面积 = (50-10) × (50-10) = 40 × 40 = 1600
B 的面积 = (70-30) × (70-30) = 40 × 40 = 1600
并集面积 = A面积 + B面积 - 交集面积 = 1600 + 1600 - 400 = 2800

3. IoU = 交集面积 / 并集面积 = 400 / 2800 = 1/7 ≈ 0.142857

答案：IoU = 1/7 ≈ 0.142857

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):
    """
    计算标签平滑后的交叉熵损失
    logits: 模型输出，形状 (batch_size, num_classes)
    labels: 真实标签，形状 (batch_size,)
    epsilon: 平滑因子
    """
    num_classes = logits.size(-1)
    # 构造平滑后的标签分布
    smooth_labels = torch.full_like(logits, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    # 计算 log softmax
    log_probs = F.log_softmax(logits, dim=-1)
    # 交叉熵 = -sum(p * log(q))
    loss = -torch.sum(smooth_labels * log_probs, dim=-1).mean()
    return loss

# 测试
logits = torch.randn(4, 10)
labels = torch.tensor([1, 3, 5, 7])
loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
print("标签平滑交叉熵损失:", loss.item())

# 对比普通交叉熵
ce_loss = F.cross_entropy(logits, labels)
print("普通交叉熵损失:", ce_loss.item())

标签平滑交叉熵损失: 3.096799373626709
普通交叉熵损失: 3.1340835094451904
